# Borrador histórico

Este cuaderno conserva una exploración previa. Para la entrega y para la presentación usamos [Proyecto_Final_B_Portafolio_Cine.ipynb](../Proyecto_Final_B_Portafolio_Cine.ipynb). Los montos, filtros y propuestas de este borrador pueden ser diferentes y no se deben combinar con los de la versión final.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 25)
raiz = Path.cwd().resolve()
if not (raiz / 'datos' / 'peliculas.csv').exists():
    raiz = raiz.parent
assert (raiz / 'datos' / 'peliculas.csv').exists()
salidas = raiz / 'salidas'
salidas.mkdir(exist_ok=True)
tipos_peliculas = {'id_pelicula':'int64', 'titulo':'string', 'anio_estreno':'int64',
                   'fecha_estreno':'string', 'genero':'string',
                   'clasificacion':'string', 'pais_origen':'string',
                   'es_secuela':'string', 'director':'string', 'guionista':'string',
                   'protagonista':'string', 'productora':'string'}
raw = pd.read_csv(raiz / 'datos' / 'peliculas.csv', dtype=tipos_peliculas)
ipc = pd.read_csv(raiz / 'datos' / 'ipc_eeuu.csv',
                  dtype={'anio':'int64', 'ipc_eeuu':'float64'})
# Hay fechas ISO y DD/MM/AAAA: format='mixed' y dayfirst conservan ambas.
raw['fecha_parseada'] = pd.to_datetime(raw.fecha_estreno, format='mixed',
                                       dayfirst=True, errors='coerce')
print('Filas y columnas originales:', (len(raw),len(raw.columns)-1), '| IPC:', ipc.shape)
print('Período:', raw.anio_estreno.min(), '-', raw.anio_estreno.max())
display(raw.head())
display(ipc.head())
esquema = pd.concat([raw.dtypes.rename('tipo').to_frame().assign(archivo='peliculas'),
                     ipc.dtypes.rename('tipo').to_frame().assign(archivo='ipc')])
display(esquema)


# Arroyo Capital: tesis de inversión en cine

**Proyecto final B · Programación Analítica II · Nicolás Gómez**

**Pregunta del comité:** cómo distribuir USD 500 millones entre tipos de película, considerando retorno, probabilidad de fracaso y evidencia fuera de muestra.

Este notebook contiene la auditoría, las decisiones de limpieza, el análisis y una propuesta ejecutiva. Utiliza los archivos entregados en `datos/` y deja los gráficos en `salidas/`. Ejecutar las celdas de arriba hacia abajo.

**Herramientas del curso utilizadas:** funciones y ciclos (clases 1–3), pandas para limpieza, agrupaciones y cruces (clases 4–5), gráficos Plotly (clase 7), auditoría de faltantes, escalas y fechas y generación aleatoria con NumPy (clase 10), y creación de variables y tramos (clase 12). La simulación combina esas herramientas para responder a la opción expresamente permitida en la consigna.


## 1. Supuestos y método

La boletería mundial es recaudación bruta, no ganancia del estudio. El criterio comercial de la consigna es **recaudación/presupuesto ≥ 2,5**. Para aproximar el retorno del capital suponemos que el estudio recibe 50% de la boletería y gasta 50% adicional del presupuesto en marketing: `retorno = recaudación/(3 × presupuesto) − 1`. Si marketing equivale al 100% del presupuesto, el divisor pasa a **4**. La regla comercial de 2,5x y el punto de equilibrio de este modelo (3x) son indicadores distintos. No se incluyen ingresos posteriores por streaming o licencias.

Todos los montos se convierten a dólares de 2020 con `IPC_2020/IPC_año`. Se diseñan los criterios con estrenos hasta **2014** y se prueban sin cambiarlos en **2015–2020**. Para reducir el efecto de éxitos extraordinarios, la simulación limita cada múltiplo a 10x y muestra sensibilidad con 8x. La mediana y la media sin límite también se reportan.


In [ ]:
cols = ['presupuesto_usd', 'recaudacion_mundial_usd', 'duracion_minutos',
        'puntaje_imdb', 'cantidad_votos']
exactas = raw.drop(columns='fecha_parseada').duplicated()
ids_en_conflicto = raw.loc[~exactas].duplicated('id_pelicula').sum()
assert ids_en_conflicto == 0, 'Hay ID repetidos con datos distintos: revisar antes de borrar'
print('Duplicados exactos:', exactas.sum(), '| ID conflictivos:', ids_en_conflicto)
display(raw.loc[raw.presupuesto_usd.isna(),
                ['id_pelicula','titulo','anio_estreno','genero','presupuesto_usd']].head(8))
categoricas=['genero','clasificacion','pais_origen','es_secuela',
             'director','guionista','protagonista','productora']
display(raw.select_dtypes(include='number').describe().T.round(2))
print('Fechas sin parsear:',raw.fecha_parseada.isna().sum(),
      '| año de fecha distinto del declarado:',
      ((raw.fecha_parseada.dt.year != raw.anio_estreno)&raw.fecha_parseada.notna()).sum())
display(raw.loc[(raw.fecha_parseada.dt.year != raw.anio_estreno)&raw.fecha_parseada.notna(),
                ['titulo','anio_estreno','fecha_estreno','fecha_parseada']].head(8))
for c in ['presupuesto_usd', 'recaudacion_mundial_usd',
          'duracion_minutos', 'puntaje_imdb']:
    print(f'{c}: no positivos={(raw[c] <= 0).sum()}, máximo={raw[c].max():,.0f}')
for c in ['genero', 'clasificacion', 'pais_origen']:
    print(f'\n{c}: {raw[c].nunique(dropna=False)} variantes')


## 2. Auditoría de calidad

Primero contamos duplicados, faltantes, centinelas y variantes de categorías. La calidad del presupuesto es central: casi tres de cada diez filas no traen ese dato. Las fechas se interpretan con sus dos formatos; 1.297 registros depurados tienen un año de fecha distinto del año declarado. Para IPC, cobertura y separación temporal se usa `anio_estreno` como año de referencia del archivo, y la discrepancia queda documentada.


In [ ]:
mapa_genero = {'accion':'Acción', 'biografia':'Biografía',
               'animacion':'Animación', 'ciencia ficcion':'Ciencia ficción'}
mapa_clasificacion = {'atp':'ATP', 'apta todo publico':'ATP',
                      'supervision parental':'Supervisión parental',
                      'supervisión parental':'Supervisión parental',
                      'mayores de 13':'Mayores de 13',
                      'mayores de 17':'Mayores de 17',
                      'solo adultos':'Solo adultos',
                      'sin clasificar':'Sin clasificar'}
# Sólo se eliminan copias exactas; comprobamos arriba que no hay ID conflictivos.
datos = raw.loc[~exactas].copy()
calidad=[]
def registrar(accion, afectados, base=len(datos), decision=''):
    calidad.append({'accion':accion,'filas_afectadas':int(afectados),
                    'porcentaje_base':round(100*afectados/base,2),
                    'decision':decision})
registrar('Duplicados exactos eliminados', exactas.sum(),len(raw),
         'No representan películas adicionales')
antes = datos[['genero','clasificacion','pais_origen']].copy()
datos['genero'] = (datos.genero.str.strip().str.lower()
                   .replace(mapa_genero).str.capitalize())
datos['clasificacion'] = (datos.clasificacion.str.strip().str.lower()
                           .replace(mapa_clasificacion).fillna('No informado'))
datos['pais_origen'] = datos.pais_origen.str.strip().replace(
    {'EE.UU.': 'Estados Unidos', 'UK': 'Reino Unido',
     'Canada': 'Canadá', 'Japon': 'Japón'})
for c in antes:
    registrar('Categoría normalizada: '+c,(antes[c].fillna('') != datos[c].fillna('')).sum(),
             decision='Unificar etiquetas equivalentes')
for c in ['anio_estreno'] + cols:
    nuevo_nulo=(datos[c].notna() & pd.to_numeric(datos[c],errors='coerce').isna()).sum()
    assert nuevo_nulo == 0, f'Conversión numérica perdió valores: {c}'
    datos[c] = pd.to_numeric(datos[c], errors='coerce')
registrar('Presupuesto no positivo → faltante',datos.presupuesto_usd.le(0).sum(),
         decision='No admite cálculo de retorno')
datos.loc[datos.presupuesto_usd.le(0), 'presupuesto_usd'] = np.nan
registrar('Recaudación negativa → faltante',datos.recaudacion_mundial_usd.lt(0).sum(),
         decision='Valor imposible')
datos.loc[datos.recaudacion_mundial_usd.lt(0), 'recaudacion_mundial_usd'] = np.nan
registrar('Duración fuera de 50–300 min → faltante',(~datos.duracion_minutos.between(50,300)&datos.duracion_minutos.notna()).sum(),
         decision='Valor fuera del rango plausible de largometrajes')
datos.loc[~datos.duracion_minutos.between(50, 300), 'duracion_minutos'] = np.nan
registrar('Puntaje fuera de 1–10 → faltante',(~datos.puntaje_imdb.between(1,10)&datos.puntaje_imdb.notna()).sum(),
         decision='Escala IMDb inválida')
datos.loc[~datos.puntaje_imdb.between(1, 10), 'puntaje_imdb'] = np.nan
registrar('Votos no positivos → faltante',datos.cantidad_votos.le(0).sum(),
         decision='Conteo inválido')
datos.loc[datos.cantidad_votos.le(0), 'cantidad_votos'] = np.nan
fecha = datos.fecha_parseada
print('Filas tras quitar duplicados:', len(datos))
print('Año de fecha distinto del año declarado:',
      ((fecha.dt.year != datos.anio_estreno) & fecha.notna()).sum())
print('Presupuesto ausente o centinela:', datos.presupuesto_usd.isna().sum())
print('Recaudación ausente o negativa:', datos.recaudacion_mundial_usd.isna().sum())
display(pd.DataFrame(calidad))
display(pd.DataFrame({'antes':raw[categoricas[:4]].nunique(dropna=False),
                      'despues':datos[categoricas[:4]].nunique(dropna=False)}))
display(pd.DataFrame({
    'faltantes_antes_pct':(100*raw.isna().mean()).round(2),
    'faltantes_despues_pct':(100*datos.isna().mean()).round(2)}))


In [ ]:
datos['multiplo_crudo'] = datos.recaudacion_mundial_usd / datos.presupuesto_usd
datos['escala_ambigua'] = (datos.presupuesto_usd.lt(1_000_000) &
                           datos.multiplo_crudo.gt(100))
print('Escala ambigua:', datos.escala_ambigua.sum())
print('Presupuesto positivo < USD 1 millón:',
      datos.presupuesto_usd.between(1, 999_999).sum())
display(datos.loc[datos.escala_ambigua,
                  ['titulo','presupuesto_usd','recaudacion_mundial_usd','multiplo_crudo']]
        .sort_values('multiplo_crudo', ascending=False).head(10).round(1))
display(datos.loc[datos.presupuesto_usd.between(1,999_999)&
                  ~datos.escala_ambigua,
                  ['titulo','presupuesto_usd','recaudacion_mundial_usd','multiplo_crudo']]
        .head(10).round(1))
registrar('Presupuesto < USD 1 M excluido de tesis',
         datos.presupuesto_usd.between(1,999_999).sum(),
         decision='Escala ambigua; excluye también micro películas genuinas')
print('Porcentaje excluido por presupuesto < USD 1 M:',
      round(100*datos.presupuesto_usd.between(1,999_999).mean(),2))


### Presupuestos con escala ambigua

Hay películas cuyo presupuesto aparece en miles de dólares mientras su recaudación está en decenas o cientos de millones. Algunas son micro presupuestos genuinos, por lo que multiplicar automáticamente por 1.000 sería injustificado. Marcamos como ambiguas las observaciones con presupuesto menor a USD 1 millón y múltiplo mayor a 100. Para la tesis de inversión exigimos presupuesto registrado de al menos USD 1 millón corriente; esta restricción también excluye micro películas genuinas, una limitación de alcance explícita.


In [ ]:
assert len(ipc) == ipc.anio.nunique() and ipc.ipc_eeuu.gt(0).all()
ipc_2020 = ipc.loc[ipc.anio.eq(2020), 'ipc_eeuu'].iloc[0]
datos = datos.merge(ipc.rename(columns={'anio':'anio_estreno'}),
                    on='anio_estreno', how='left')
assert len(datos) == len(raw)-exactas.sum()
assert datos.ipc_eeuu.notna().all()
factor = ipc_2020 / datos.ipc_eeuu
datos['presupuesto_2020'] = datos.presupuesto_usd * factor
datos['recaudacion_2020'] = datos.recaudacion_mundial_usd * factor
datos['multiplo'] = datos.recaudacion_2020 / datos.presupuesto_2020
datos['roi_salas'] = datos.multiplo/3 - 1
datos['recupero_25'] = datos.multiplo.ge(2.5).where(datos.multiplo.notna(),pd.NA).astype('boolean')
datos['fracaso'] = datos.multiplo.lt(2.5).where(datos.multiplo.notna(),pd.NA).astype('boolean')
datos['tramo'] = pd.cut(datos.presupuesto_2020/1e6,
                        [0, 10, 30, 70, np.inf],
                        labels=['<10 M','10–30 M','30–70 M','70+ M'], right=False)
datos['epoca'] = pd.cut(datos.anio_estreno,
                        [1979,1989,1999,2009,2014,2020],
                        labels=['1980s','1990s','2000s','2010–14','2015–20'])
universo = datos.loc[datos.presupuesto_usd.ge(1_000_000) &
                     datos.recaudacion_mundial_usd.notna()].copy()
registrar('Sin retorno calculable en la tesis',len(datos)-len(universo),
         decision='Presupuesto >= USD 1 M y recaudación informada')
print('Porcentaje sin retorno calculable en la tesis:',
      round(100*(len(datos)-len(universo))/len(datos),2))
universo['multiplo_tope10'] = universo.multiplo.clip(upper=10)
train = universo.loc[universo.anio_estreno.le(2014)].copy()
test = universo.loc[universo.anio_estreno.between(2015,2020)].copy()
print('Universo:',len(universo),'| entrenamiento:',len(train),'| prueba:',len(test))
print('Industria en entrenamiento: mediana',round(train.multiplo.median(),2),
      'media',round(train.multiplo.mean(),2),
      'media hasta 10x',round(train.multiplo_tope10.mean(),2),
      'fracaso',f'{train.fracaso.mean():.1%}')


## 3. Inflación y retorno

El cociente recaudación/presupuesto es el mismo en dólares corrientes y reales. La corrección por IPC permite comparar el tamaño de proyectos de distintas décadas y definir tramos en USD de 2020.


In [ ]:
evolucion = datos.groupby('anio_estreno').agg(
    n=('id_pelicula','size'),
    n_recaudacion=('recaudacion_mundial_usd','count'),
    recaudacion_nominal_m=('recaudacion_mundial_usd','median'),
    recaudacion_real_m=('recaudacion_2020','median')).reset_index()
evolucion['sin_recaudacion_pct'] = (100*(1-evolucion.n_recaudacion/evolucion.n)).round(1)
evolucion[['recaudacion_nominal_m','recaudacion_real_m']] /= 1e6
display(evolucion.loc[evolucion.anio_estreno.isin(
    [1980,1985,1990,1995,2000,2005,2010,2015,2018,2019,2020])].round(2))
print('Años presentes:',evolucion.anio_estreno.nunique(),
      '| mínimo de películas/año:',evolucion.n.min(),
      '| máximo:',evolucion.n.max())
print('Títulos de 2020 sin recaudación registrada:')
display(datos.loc[datos.anio_estreno.eq(2020)&datos.recaudacion_mundial_usd.isna(),
                  ['titulo','genero','presupuesto_usd','recaudacion_mundial_usd']])


### ¿Creció la industria?

La mediana nominal y la mediana ajustada por IPC se calculan sobre las mismas películas de cada año. Un valor de 1980 se multiplica por `IPC_2020 / IPC_1980`; así todos los montos quedan en poder de compra de 2020. La tabla muestra también cuántas películas tienen recaudación informada. **La línea de tendencia termina en 2019**, porque 2020 tiene una cobertura muy diferente; sus datos se auditan por separado.


In [ ]:
# PLOTLY_OUTPUT
evolucion_grafico = evolucion.loc[evolucion.anio_estreno.lt(2020)].copy()
figura_interactiva = go.Figure()
figura_interactiva.add_trace(go.Scatter(
    x=evolucion_grafico.anio_estreno,y=evolucion_grafico.recaudacion_nominal_m,
    customdata=evolucion_grafico.n_recaudacion,
    name='Dólares corrientes',mode='lines+markers'))
figura_interactiva.add_trace(go.Scatter(
    x=evolucion_grafico.anio_estreno,y=evolucion_grafico.recaudacion_real_m,
    customdata=evolucion_grafico.n_recaudacion,
    name='Dólares de 2020',mode='lines+markers'))
figura_interactiva.update_traces(hovertemplate=(
    '%{fullData.name}<br>Año: %{x}<br>Mediana: USD %{y:.1f} M'
    '<br>Películas con recaudación: %{customdata}<extra></extra>'))
figura_interactiva.update_layout(
    title='Recaudación mediana nominal y real por año (1980–2019)',
    xaxis_title='Año de estreno',
    yaxis_title='Recaudación mediana (millones USD)')
figura_interactiva.write_html(salidas/'02_evolucion_interactiva.html',include_plotlyjs=True)
display(figura_interactiva)


In [ ]:
def peor_decil(s):
    return s.loc[s.le(s.quantile(.10))].mean()

def resumen(grupo, columna, minimo=1):
    tabla = grupo.groupby(columna, observed=True).agg(
        n=('id_pelicula','size'), mediana_x=('multiplo','median'),
        media_x=('multiplo','mean'), media_tope10_x=('multiplo_tope10','mean'),
        roi_medio_pct=('roi_salas','mean'),roi_mediano_pct=('roi_salas','median'),
        roi_peor_decil_pct=('roi_salas',peor_decil),
        fracaso_pct=('fracaso','mean')).reset_index()
    tabla['fracaso_pct'] *= 100
    for col in ['roi_medio_pct','roi_mediano_pct','roi_peor_decil_pct']:
        tabla[col] *= 100
    return tabla.loc[tabla.n.ge(minimo)].sort_values('n',ascending=False).round(2)

for variable, minimo in [('genero',20),('es_secuela',1)]:
    print('\n' + variable.upper())
    display(resumen(train,variable,minimo).head(15))
display(train.sort_values('multiplo',ascending=False).head(12)[
    ['titulo','anio_estreno','genero','presupuesto_usd',
     'recaudacion_mundial_usd','multiplo','roi_salas']].round(2))


**Conclusión:** entre 1980 y 2019, la mediana nominal pasa de USD 15,0 M a USD 27,5 M, mientras la mediana real pasa de USD 47,1 M a USD 27,9 M de 2020. La inflación cambia la lectura de la tendencia. La cifra de 2020, USD 166,5 M, es la mediana de solo **15 películas con recaudación entre 25 registradas**: faltan 10 (40%), varias de bajo presupuesto. Por esa selección y la escasa cobertura, no se interpreta como un aumento de la industria ni se une a la línea de 2019.


## 4. Rendimiento y pérdidas por segmento

Las tablas muestran cantidad de películas, mediana, media, media limitada a 10x y proporción por debajo de 2,5x. Las tablas principales comparan género y franquicia; los tramos de presupuesto se estudian en los arquetipos de inversión.


In [ ]:
# PLOTLY_OUTPUT
histograma = train[['multiplo']].copy()
histograma['multiplo'] = histograma.multiplo.clip(upper=15)
figura_interactiva = px.histogram(histograma,x='multiplo',nbins=30,
    color_discrete_sequence=['#17365d'])
figura_interactiva.update_traces(hovertemplate=(
    'Múltiplo: %{x:.1f}x<br>Películas: %{y:,}<extra></extra>'))
figura_interactiva.add_vline(x=2.5, line_dash='dash', line_color='#db7c26',
                             annotation_text='Umbral comercial 2,5x')
figura_interactiva.add_vline(x=train.multiplo.median(),
                             line_color='#2b806a', annotation_text='Mediana')
figura_interactiva.update_layout(
    title='Distribución del múltiplo de boletería (1980–2014)',
    xaxis_title='Recaudación / presupuesto (15x agrupa valores mayores)',
    yaxis_title='Películas', bargap=0, height=500)
figura_interactiva.write_html(salidas/'01_distribucion_interactiva.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'01_distribucion_interactiva.html')


Observaciones: la mediana de la industria invertible es 1,69x y 64,2% de las películas no llega a 2,5x. En terror la media sin límite es 7,05x, pero la mediana es 2,75x y la media limitada a 10x es 3,78x. Los éxitos extremos no representan el resultado típico.

La tabla de películas con mayor múltiplo muestra casos como *The Devil Inside* y *Saw* que explican parte de esa diferencia. Sus presupuestos son de al menos USD 1 M y no fueron marcados por la regla de escala ambigua; no hay evidencia interna suficiente para reemplazarlos. El tope de 10x se usa en la simulación como prueba de robustez, sin alterar las observaciones originales.


### Distribución del retorno: gráfico interactivo

Pasá el mouse sobre cada barra para ver el múltiplo y la cantidad de películas. Los valores mayores a 15x se agrupan en la última barra. [Abrir el gráfico en el navegador](../salidas/01_distribucion_interactiva.html).


In [ ]:
# PLOTLY_OUTPUT
generos_hover = resumen(train, 'genero', 20).sort_values('mediana_x')
figura_interactiva = px.bar(
    generos_hover, x='mediana_x', y='genero', orientation='h',
    custom_data=['n', 'media_x', 'media_tope10_x', 'fracaso_pct'],
    color_discrete_sequence=['#17365d'],
    labels={'mediana_x': 'Mediana de recaudación / presupuesto', 'genero': 'Género'},
    title='Retorno mediano por género (1980–2014)')
figura_interactiva.update_traces(hovertemplate=(
    '<b>%{y}</b><br>Mediana: %{x:.2f}x<br>Media: %{customdata[1]:.2f}x'
    '<br>Media hasta 10x: %{customdata[2]:.2f}x'
    '<br>Fracaso comercial: %{customdata[3]:.1f}%'
    '<br>Películas: %{customdata[0]:.0f}<extra></extra>'))
figura_interactiva.update_layout(height=510)
figura_interactiva.write_html(salidas/'05_generos_interactivo.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'05_generos_interactivo.html')


**Conclusión:** la distribución se concentra en múltiplos bajos y presenta una cola de éxitos extraordinarios. Por eso la media sola sobrestima el resultado típico; se reportan también mediana y riesgo de pérdida.


### Géneros: gráfico interactivo

Pasá el mouse sobre cada barra para ver mediana, media, media limitada, número de películas y porcentaje de fracaso. [Abrir el gráfico en el navegador](../salidas/05_generos_interactivo.html).


In [ ]:
correlaciones = train[['puntaje_imdb','roi_salas']].dropna().corr()
# PLOTLY_OUTPUT
figura_interactiva = px.imshow(correlaciones,text_auto='.3f',zmin=-1,zmax=1,
    color_continuous_scale='RdBu',aspect='auto',
    labels={'color':'Correlación de Pearson'},
    title='Correlación entre puntaje IMDb y ROI de salas')
figura_interactiva.update_xaxes(title='Variable')
figura_interactiva.update_yaxes(title='Variable')
figura_interactiva.write_html(salidas/'03_imdb_roi_interactivo.html',
                              include_plotlyjs=True)
display(figura_interactiva)


**Conclusión:** las diferencias entre media y mediana por género indican que unos pocos éxitos elevan el promedio. Los géneros se comparan por su mediana y por la frecuencia de fracaso, además de la media.


### IMDb y retorno

El puntaje de IMDb se conoce después del estreno: esta relación es descriptiva y no permite seleccionarlo como criterio previo de inversión.


In [ ]:
segmentos = train.groupby(['genero','tramo','es_secuela'],observed=True).agg(
    n=('id_pelicula','size'),mediana_x=('multiplo','median'),
    retorno_robusto_x=('multiplo_tope10','mean'),
    prob_fracaso=('fracaso','mean'),
    mediana_roi=('roi_salas','median'),
    roi_peor_decil=('roi_salas',peor_decil),
    presupuesto_mediano=('presupuesto_2020','median')).reset_index()
segmentos = segmentos.loc[segmentos.n.ge(30)].copy()
segmentos['arquetipo'] = (segmentos.genero+' | '+segmentos.tramo.astype(str)
                          +' | secuela='+segmentos.es_secuela)
segmentos['dominado'] = [
    bool(((segmentos.mediana_roi>=fila.mediana_roi)&
          (segmentos.prob_fracaso<=fila.prob_fracaso)&
          ((segmentos.mediana_roi>fila.mediana_roi)|
           (segmentos.prob_fracaso<fila.prob_fracaso))).any())
    for _,fila in segmentos.iterrows()]
segmentos['mediana_roi_pct']=100*segmentos.mediana_roi
segmentos['roi_peor_decil_pct']=100*segmentos.roi_peor_decil
segmentos['fracaso_pct']=100*segmentos.prob_fracaso
display(segmentos.sort_values('mediana_roi',ascending=False)[
    ['arquetipo','n','mediana_roi_pct','roi_peor_decil_pct',
     'fracaso_pct','dominado']].round(2))


**Conclusión:** la correlación mide asociación lineal, no causalidad. Los grandes éxitos pueden influir mucho en Pearson; el puntaje tampoco estaba disponible cuando se habría decidido financiar la película.


## 5. Plano de riesgo y retorno

Se comparan arquetipos de género, tramo de presupuesto y franquicia con al menos 30 películas en entrenamiento. El eje horizontal mide la probabilidad histórica de fracaso comercial. El vertical muestra la media limitada a 10x. El tamaño del punto representa el número de observaciones.


In [ ]:
# PLOTLY_OUTPUT
riesgo_hover = segmentos.assign(
    fracaso_pct=100*segmentos.prob_fracaso,
    presupuesto_mediano_m=segmentos.presupuesto_mediano/1e6)
figura_interactiva = px.scatter(
    riesgo_hover, x='fracaso_pct', y='retorno_robusto_x',
    size='n', color='genero',
    custom_data=['arquetipo','n','mediana_x','presupuesto_mediano_m'],
    labels={'fracaso_pct': 'Probabilidad de fracaso comercial (%)',
            'retorno_robusto_x': 'Media del múltiplo limitada a 10x',
            'genero': 'Género'},
    title='Riesgo y retorno por arquetipo (1980–2014)')
figura_interactiva.update_traces(hovertemplate=(
    '<b>%{customdata[0]}</b><br>Fracaso: %{x:.1f}%'
    '<br>Media hasta 10x: %{y:.2f}x<br>Mediana: %{customdata[2]:.2f}x'
    '<br>Presupuesto mediano: USD %{customdata[3]:.1f} M'
    '<br>Películas: %{customdata[1]:.0f}<extra></extra>'))
figura_interactiva.add_hline(y=3, line_dash='dash', line_color='#db7c26')
figura_interactiva.update_layout(height=620)
figura_interactiva.write_html(salidas/'06_riesgo_retorno_interactivo.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'06_riesgo_retorno_interactivo.html')


### Riesgo y retorno: gráfico interactivo

Pasá el mouse sobre cada burbuja para identificar el arquetipo, la cantidad de películas, el presupuesto mediano y sus medidas de retorno y fracaso. [Abrir el gráfico en el navegador](../salidas/06_riesgo_retorno_interactivo.html).


In [ ]:
datos['sin_presupuesto']=datos.presupuesto_usd.isna()
por_genero=datos.groupby('genero').agg(
    n=('id_pelicula','size'),tasa=('sin_presupuesto','mean')).reset_index()
por_genero=por_genero.loc[por_genero.n.ge(50)]
por_epoca=datos.groupby('epoca',observed=True).agg(
    n=('id_pelicula','size'),tasa=('sin_presupuesto','mean')).reset_index()
datos['decada']=(datos.anio_estreno//10)*10
por_decada=datos.groupby('decada').agg(
    n=('id_pelicula','size'),tasa=('sin_presupuesto','mean')).reset_index()
por_pais=datos.groupby('pais_origen').agg(
    n=('id_pelicula','size'),tasa=('sin_presupuesto','mean')).reset_index()
por_pais=por_pais.loc[por_pais.n.ge(30)]
print('Por género:')
display(por_genero.assign(tasa=lambda d:(100*d.tasa).round(1)).sort_values('tasa',ascending=False))
print('Por década:')
display(por_decada.assign(tasa=lambda d:(100*d.tasa).round(1)))
print('Por país (n ≥ 30):')
display(por_pais.assign(tasa=lambda d:(100*d.tasa).round(1)).sort_values('tasa',ascending=False))
print('Recaudación mediana en USD millones de 2020:')
medianas_recaudacion = datos.groupby('sin_presupuesto').recaudacion_2020.median()/1e6
print('Con presupuesto: USD',round(medianas_recaudacion.loc[False],2),'M;',
      'sin presupuesto: USD',round(medianas_recaudacion.loc[True],2),'M')
comparables=datos.loc[datos.recaudacion_mundial_usd.notna() &
                      (datos.presupuesto_usd.isna()|datos.presupuesto_usd.ge(1_000_000))]
conocidos=comparables.loc[comparables.presupuesto_usd.notna()]
f=conocidos.fracaso.sum();m=comparables.presupuesto_usd.isna().sum()
print('Fracaso observado:',f'{f/len(conocidos):.1%}')
print('Cota si todos los faltantes fueran éxitos:',f'{f/len(comparables):.1%}')
print('Cota si todos fueran fracasos:',f'{(f+m)/len(comparables):.1%}')


**Conclusión:** un retorno central alto no garantiza bajo riesgo. La tabla anterior identifica los segmentos dominados con mediana de ROI y probabilidad de fracaso; el gráfico permite inspeccionar también la media limitada a 10x y el tamaño de cada muestra.


## 6. Sesgo por presupuestos faltantes

Las películas sin presupuesto no forman necesariamente un grupo aleatorio. Medimos la tasa de faltantes por género y época, y comparamos su recaudación con la de las películas que sí tienen presupuesto. No imputamos valores para seleccionar inversiones. Las cotas extremas de fracaso suponiendo que todos los faltantes fueran éxitos o fracasos muestran la incertidumbre que deja esa decisión.


In [ ]:
# PLOTLY_OUTPUT
faltantes_hover = por_genero.assign(tasa_pct=100*por_genero.tasa).sort_values('tasa_pct')
figura_interactiva = px.bar(
    faltantes_hover, x='tasa_pct', y='genero', orientation='h',
    custom_data=['n'], color_discrete_sequence=['#db7c26'],
    labels={'tasa_pct': 'Sin presupuesto (%)', 'genero': 'Género'},
    title='Presupuestos faltantes por género')
figura_interactiva.update_traces(hovertemplate=(
    '<b>%{y}</b><br>Sin presupuesto: %{x:.1f}%'
    '<br>Películas: %{customdata[0]:.0f}<extra></extra>'))
figura_interactiva.update_layout(height=500)
figura_interactiva.write_html(salidas/'07_faltantes_interactivo.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'07_faltantes_interactivo.html')


**Efecto sobre la conclusión:** falta presupuesto en 42,5% de los dramas y 17,0% de las películas de acción. La recaudación mediana de las películas sin presupuesto es USD 6,69 M de 2020, frente a USD 53,90 M de las demás. La tasa de fracaso observada en el universo invertible es 62,0%; bajo supuestos extremos sobre los faltantes, puede estar entre 44,3% y 72,8%. La muestra con retorno calculable probablemente favorece películas de mayor escala.


### Presupuestos faltantes: gráfico interactivo

Cada barra muestra la tasa de faltantes y, al pasar el mouse, el número de películas de ese género. [Abrir el gráfico en el navegador](../salidas/07_faltantes_interactivo.html).


In [ ]:
# PLOTLY_OUTPUT
epocas_hover = por_epoca.assign(tasa_pct=100*por_epoca.tasa,
                                epoca_texto=por_epoca.epoca.astype(str))
figura_interactiva = px.bar(
    epocas_hover, x='epoca_texto', y='tasa_pct', custom_data=['n'],
    color_discrete_sequence=['#db7c26'],
    labels={'epoca_texto':'Época','tasa_pct':'Sin presupuesto (%)'},
    title='Presupuestos faltantes por época')
figura_interactiva.update_traces(hovertemplate=(
    '<b>%{x}</b><br>Sin presupuesto: %{y:.1f}%'
    '<br>Películas: %{customdata[0]:.0f}<extra></extra>'))
figura_interactiva.update_layout(height=480)
figura_interactiva.write_html(salidas/'09_faltantes_epoca_interactivo.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'09_faltantes_epoca_interactivo.html')


**Conclusión:** la tasa de presupuestos faltantes varía entre géneros. Excluir esos títulos puede cambiar tanto el nivel estimado de fracaso como la comparación entre tipos de películas.


### Presupuestos faltantes por época: gráfico interactivo

El hover muestra la tasa de faltantes y la cantidad de películas de cada período. [Abrir el gráfico en el navegador](../salidas/09_faltantes_epoca_interactivo.html).


In [ ]:
FONDO=500_000_000
cartera=pd.DataFrame([('Terror','<10 M','No',.32),
                       ('Terror','10–30 M','No',.25),
                       ('Animación','30–70 M','No',.28),
                       ('Comedia','<10 M','No',.15)],
                      columns=['genero','tramo','es_secuela','peso'])
cartera['tramo']=cartera.tramo.astype(str)
segmentos['tramo']=segmentos.tramo.astype(str)
cartera=cartera.merge(segmentos,on=['genero','tramo','es_secuela'])
assert len(cartera)==4
assert abs(cartera.peso.sum()-1)<1e-12 and cartera.n.ge(30).all()
cartera['asignacion_millones']=cartera.peso*FONDO/1e6
cartera['proyectos']=(cartera.peso*FONDO/
                      (1.5*cartera.presupuesto_mediano)).round().astype(int)
assert cartera.proyectos.ge(1).all()
cartera['capital_por_proyecto_m']=cartera.asignacion_millones/cartera.proyectos
cartera['costo_tipico_m']=1.5*cartera.presupuesto_mediano/1e6
cartera['retorno_base_pct']=100*(cartera.retorno_robusto_x/3-1)
display(cartera[['arquetipo','n','asignacion_millones','proyectos',
                 'capital_por_proyecto_m','costo_tipico_m','mediana_x',
                 'retorno_robusto_x','prob_fracaso','retorno_base_pct']].round(2))
print('Total asignado:',cartera.asignacion_millones.sum(),'M USD')
print('Proyectos aproximados:',cartera.proyectos.sum())


**Conclusión:** el registro de presupuestos cambia entre períodos. Esa variación limita cualquier interpretación de una tendencia histórica basada solo en películas con presupuesto conocido.


## 7. Asignación propuesta

Los cuatro arquetipos se eligieron usando solo 1980–2014 y tienen al menos 30 observaciones. Los pesos suman USD 500 millones: **32% terror <10 M**, **25% terror 10–30 M**, **28% animación 30–70 M** y **15% comedia <10 M**, todos no secuelas. Los rangos son de presupuesto de producción en USD de 2020. La mezcla reduce la concentración en terror de bajo presupuesto, cuyo retorno robusto es el mayor, pero tiene solo 48 películas de entrenamiento. El número de proyectos surge de dividir la asignación por `1,5 × presupuesto mediano` y redondear; cada bloque invierte su asignación completa en partes iguales.


In [ ]:
def simular(datos_fuente, divisor=3, tope=10, repeticiones=20_000, semilla=2026):
    rng=np.random.default_rng(semilla)
    retornos=np.zeros(repeticiones)
    for _,f in cartera.iterrows():
        grupo=datos_fuente.loc[(datos_fuente.genero==f.genero)&
                               (datos_fuente.tramo==f.tramo)&
                               (datos_fuente.es_secuela==f.es_secuela),'multiplo']
        if grupo.empty:
            raise ValueError('Sin observaciones: '+f.arquetipo)
        sorteos=rng.choice(grupo.clip(upper=tope),
                           size=(repeticiones,int(f.proyectos)),replace=True)
        retornos+=f.peso*(sorteos.mean(axis=1)/divisor-1)
    return pd.Series(retornos)

base=simular(train)
alto=simular(train,divisor=4)
print('Escenario base: retorno medio',f'{base.mean():.1%}',
      '| ganancia esperada',f'USD {base.mean()*500:.1f} M',
      '| probabilidad de pérdida',f'{(base<0).mean():.1%}')
print('Percentiles de retorno 5/50/95:',
      [f'{base.quantile(q):.1%}' for q in [.05,.5,.95]])
print('Pérdida media del peor 5%:',
      f'USD {-base[base<=base.quantile(.05)].mean()*500:.1f} M')
print('Costos altos: retorno medio',f'{alto.mean():.1%}',
      '| probabilidad de pérdida',f'{(alto<0).mean():.1%}')


### Simulación de la cartera

Se hacen 20.000 sorteos con reemplazo de películas históricas dentro de cada arquetipo. Los proyectos se consideran independientes, una simplificación que puede subestimar una crisis común del mercado. El retorno usa la economía estimada del estudio y no representa un contrato real del fondo.


In [ ]:
# PLOTLY_OUTPUT
retornos_grafico=pd.DataFrame({'retorno_pct':base*100})
figura_interactiva=px.histogram(retornos_grafico,x='retorno_pct',nbins=50,
    color_discrete_sequence=['#17365d'])
figura_interactiva.update_traces(hovertemplate=(
    'Retorno: %{x:.1f}%<br>Simulaciones: %{y:,}<extra></extra>'))
figura_interactiva.add_vline(x=0, line_dash='dash', line_color='#db7c26')
figura_interactiva.update_layout(
    title='Distribución simulada del retorno del fondo (1980–2014)',
    xaxis_title='Retorno del fondo (%)', yaxis_title='Simulaciones',
    bargap=0, height=500)
figura_interactiva.write_html(salidas/'08_cartera_interactiva.html',
                              include_plotlyjs=True)
display(figura_interactiva)
print('HTML:', salidas/'08_cartera_interactiva.html')


Observaciones: con equilibrio a 3x, el retorno medio simulado es 27,6% y la probabilidad de perder capital es 11,0%. El percentil 5 es −7,7% y la pérdida media del peor 5% es aproximadamente USD 74,5 M. Con equilibrio a 4x, el retorno medio baja a −4,3% y la probabilidad de pérdida sube a 62,5%. La decisión depende mucho del gasto real de marketing y del reparto de boletería.


### Simulación de cartera: gráfico interactivo

Pasá el mouse sobre las barras para leer el retorno y la cantidad de simulaciones. [Abrir el gráfico en el navegador](../salidas/08_cartera_interactiva.html).


In [ ]:
filas=[]
for _,f in cartera.iterrows():
    a=train.loc[(train.genero==f.genero)&(train.tramo==f.tramo)&
                (train.es_secuela==f.es_secuela)]
    b=test.loc[(test.genero==f.genero)&(test.tramo==f.tramo)&
               (test.es_secuela==f.es_secuela)]
    filas.append({'arquetipo':f.arquetipo,'n_train':len(a),'n_test':len(b),
                  'mediana_train_x':a.multiplo.median(),
                  'mediana_test_x':b.multiplo.median(),
                  'fracaso_train_pct':100*a.fracaso.mean(),
                  'fracaso_test_pct':100*b.fracaso.mean(),
                  'retorno_test_pct':100*(b.multiplo.clip(upper=10).mean()/3-1)})
display(pd.DataFrame(filas).round(2))
prueba=simular(test)
prueba_alto=simular(test,divisor=4)
print('\nCartera con las mismas reglas en 2015–2020:')
print('Retorno ponderado observado:',f'{prueba.mean():.1%}')
print('Pérdidas en el remuestreo:',int((prueba<0).sum()),'de',len(prueba))
print('Costos altos: retorno',f'{prueba_alto.mean():.1%}',
      '| probabilidad de pérdida',f'{(prueba_alto<0).mean():.1%}')
print('Percentiles 5/95:',[f'{prueba.quantile(q):.1%}' for q in [.05,.95]])


**Conclusión:** incluso con una mezcla de proyectos hay simulaciones con pérdida. La distribución depende de los supuestos sobre costos y reparto de boletería, que se exploran en la tabla de sensibilidad.


## 8. Verificación fuera de muestra

Se mantienen arquetipos, pesos, cantidades, límite de 10x y fórmula de retorno. El resultado ponderado de 2015–2020 evalúa los **tipos** de película, no identifica las películas específicas que habría financiado el fondo. La cobertura parcial de 2020 también limita esta verificación; sus resultados no representan ese año completo.


In [ ]:
escenarios=pd.DataFrame({
    'escenario':['Train 3x tope10','Train 3x tope8','Train 4x tope10',
                 'Train 4x tope8','Prueba 3x tope10','Prueba 4x tope10'],
    'retorno_pct':[x.mean()*100 for x in
                   [base,simular(train,tope=8),alto,
                    simular(train,divisor=4,tope=8),prueba,prueba_alto]],
    'perdida_pct':[(x<0).mean()*100 for x in
                   [base,simular(train,tope=8),alto,
                    simular(train,divisor=4,tope=8),prueba,prueba_alto]]})
display(escenarios.round(1))
print('Figuras:',salidas)


**Validación:** el retorno ponderado observado en prueba es 59,6% con equilibrio a 3x. La mediana del terror <10 M pasa de 4,23x a 4,77x, pero la de animación 30–70 M baja de 2,29x a 1,27x. Terror 10–30 M tiene solo seis películas de prueba y animación siete; sus estimaciones son inestables. Ninguna pérdida en 20.000 remuestreos de esta muestra no significa riesgo futuro cero.


## 9. Sensibilidad y recomendación al comité

La propuesta es un **piloto condicionado** a verificar presupuestos, documentar la participación contractual del fondo y limitar la exposición por película. Las cifras son una tesis sobre boletería histórica, no una proyección financiera contractual. El sesgo de faltantes, la escala ambigua de algunos presupuestos, la dependencia entre películas y los ingresos fuera de salas siguen sin resolver con estos archivos.

**Dónde no invertiría inicialmente:** dramas y policiales de USD 30–70 M no secuelas presentan medianas de ROI de −63,1% y −60,8% y tasas de fracaso comercial de 69,4% y 76,8%, respectivamente, en la muestra de entrenamiento. Evitaría además presupuestos inferiores a USD 1 M hasta verificar su escala, porque los registros mezclan casos plausibles con cifras sospechosas.
